## Imports and Environment

In [1]:
# SimpleDirectoryReader is dynamic, detects file type and uses appropriate reader
from llama_index.core import VectorStoreIndex, Settings, PromptTemplate, StorageContext, SQLDatabase, SimpleDirectoryReader
from llama_index.core.utilities.sql_wrapper import SQLDatabase
from llama_index.core.query_engine import NLSQLTableQueryEngine, KnowledgeGraphQueryEngine
from llama_index.core.workflow import Workflow, StartEvent, StopEvent, step, Context, Event
from llama_index.core.retrievers import SQLRetriever
from llama_index.core.schema import TextNode
from llama_index.core.tools import QueryEngineTool
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.query_engine import NLSQLTableQueryEngine, RetrieverQueryEngine, SQLTableRetrieverQueryEngine
from llama_index.core.retrievers import SQLRetriever
from llama_index.core.objects import SQLTableNodeMapping, ObjectIndex, SQLTableSchema
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.llms.openai import OpenAI
from llama_parse import LlamaParse

from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer, util

import pandas as pd, re, ast, textwrap
from sqlalchemy import create_engine, text, inspect

from datasets import Dataset

import ragas
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
    LLMSQLEquivalence,
)
from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory
from ragas import evaluate, EvaluationDataset
from ragas.metrics import DataCompyScore

from dotenv import load_dotenv, find_dotenv
from typing import Dict, Any, Tuple, Optional, List
import numpy as np
import torch
import os
import re
import json
import csv
from tqdm import tqdm

# Project root path for Azure Sandpit environment
project_root_path = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone" 

# Change the current working directory to the project root
os.chdir(project_root_path)


# --- FIX 2: Bypass find_dotenv() and use a direct, verified path ---
dotenv_path = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/.env"

# Add a critical check to ensure the .env file exists at this path
if not os.path.exists(dotenv_path):
    raise FileNotFoundError(
        f"CRITICAL ERROR: .env file NOT FOUND at the expected path: {dotenv_path}\n"
        f"Please double-check the path you pasted into 'project_root_path'."
    )


# Load the .env file from the explicit, verified path
load_dotenv(dotenv_path=dotenv_path)

# The project root is now simply the current working directory
project_root = os.getcwd()

# --- Now, the rest of your variable loading will work correctly ---
relative_data_dir = os.getenv("SQL_DATASET_DIR")

# Add a check to make sure the variable was loaded successfully from the file
if not relative_data_dir:
    raise ValueError(
        "ERROR: 'SQL_DATASET_DIR' was not found in your .env file, or the file is empty."
    )

data_directory = os.path.join(project_root, relative_data_dir)

hf_token = os.getenv("HUGGINGFACE_TOKEN")
llama_cloud_api_key = os.getenv("LLAMA_CLOUD_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

# --- Final Verification ---
print(f"✅ Project root successfully set to: {project_root}")
print(f"✅ .env file loaded from: {dotenv_path}")
print(f"📁 Data directory set to: {data_directory}")

/anaconda/envs/py311-docext/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Project root successfully set to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/joo-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone
✅ .env file loaded from: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/.env
📁 Data directory set to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/joo-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/Datasets/SQL_Dataset


## Helper Function to Sanitize File Names

In [2]:
def sanitize_table_name(filename):
    """
    Cleans a filename to create a safe, SQL-compliant table name.
    - Converts to lowercase
    - Replaces spaces and hyphens with underscores
    - Removes all other non-alphanumeric characters (except underscores)
    """
    # Remove the .csv extension
    name = os.path.splitext(filename)[0]
    # Convert to lowercase and replace spaces/hyphens
    name = name.lower().replace(' ', '_').replace('-', '_')
    # Remove any remaining invalid characters
    name = re.sub(r'[^a-z0-9_]', '', name)
    return name

In [3]:
# Create an in-memory SQLite database
# This database exists only as long as the script is running
engine = create_engine("sqlite:///:memory:")

# --- Dynamically load all CLEANED CSVs from the 'SQL_Dataset' directory ---
# This should point to the folder where your 'run_SQL_cleaning.py' script saved the files.
sql_data_directory = "SQL_Dataset" 
table_names = [] # To keep track of the tables we create

print(f"Searching for cleaned CSV files to ingest in '{sql_data_directory}'...")

# Check if the directory exists to avoid errors
if not os.path.isdir(sql_data_directory):
    print(f"Error: The directory '{sql_data_directory}' was not found. Please ensure the cleaning script ran successfully.")
else:
    for filename in os.listdir(sql_data_directory):
        if filename.endswith(".csv"):
            try:
                file_path = os.path.join(sql_data_directory, filename)
                
                # 1. Load the already-cleaned CSV into a DataFrame
                cleaned_df = pd.read_csv(file_path)
                
                # 2. Create a clean table name from the filename
                # Example: "cleaned_m891481.csv" -> "cleaned_m891481"
                table_name = sanitize_table_name(filename)
                table_names.append(table_name)
                
                # 3. Ingest the cleaned DataFrame into the SQL database
                cleaned_df.to_sql(table_name, engine, index=False, if_exists='replace')
                
                print(f" - Successfully ingested '{filename}' into SQL table '{table_name}'")
            except Exception as e:
                print(f" - FAILED to ingest {filename}. Error: {e}")

print(f"\nIn-memory SQL database created and populated with {len(table_names)} table(s).")
sql_database = SQLDatabase(engine)


Searching for cleaned CSV files to ingest in 'SQL_Dataset'...
 - Successfully ingested 'Convicted Penal Population by Age Group (2006-2020).csv' into SQL table 'convicted_penal_population_by_age_group_2006_2020'
 - Successfully ingested 'Convicted Penal Population by Age Group (2020 onwards).csv' into SQL table 'convicted_penal_population_by_age_group_2020_onwards'
 - Successfully ingested 'Convicted Penal Population by Age Group and Offence Group (2006-2020).csv' into SQL table 'convicted_penal_population_by_age_group_and_offence_group_2006_2020'
 - Successfully ingested 'Convicted Penal Population by Age Group and Offence Group (2020 onwards).csv' into SQL table 'convicted_penal_population_by_age_group_and_offence_group_2020_onwards'
 - Successfully ingested 'Convicted Penal Population by Education Level.csv' into SQL table 'convicted_penal_population_by_education_level'
 - Successfully ingested 'Convicted Penal Population by Gender and Offence Group.csv' into SQL table 'convicted_pe

## Llama 3.1 8B Instruct

In [4]:
model_name = "meta-llama/Llama-3.1-8B-Instruct"

# Initialize the tokenizer to get the token ID for stop sequence
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
# The semicolon is the stop character, get its token ID
semicolon_token_id = tokenizer.convert_tokens_to_ids(";")

# Initialize the LLM with the correct stop condition
llm = HuggingFaceLLM(
    model_name=model_name,
    tokenizer_name=model_name,
    device_map="auto",
    model_kwargs={"token": hf_token, "dtype": torch.bfloat16},
    # Use 'eos_token_id' which is the correct parameter for this purpose
    generate_kwargs={
        "temperature": 0.1,
        "do_sample": True,
        # Stop generating as soon as it outputs a semicolon
        "eos_token_id": semicolon_token_id,
    }
)

print("HuggingFaceLLM initialized with the ';' character as the end-of-sequence token.")

Loading checkpoint shards: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [02:23<00:00, 35.99s/it]


HuggingFaceLLM initialized with the ';' character as the end-of-sequence token.


## Query Time Table Retrieval

In [5]:
inspector = inspect(engine)

# Create SQLTableNodeMapping and ObjectIndex for table retrieval
table_node_mapping = SQLTableNodeMapping(sql_database)
all_table_schema_objs = [
    SQLTableSchema(table_name=name) for name in inspector.get_table_names()
]

# Create a vector index over the table schemas for retrieval
obj_index = ObjectIndex.from_objects(
    all_table_schema_objs,
    table_node_mapping,
    index_cls=VectorStoreIndex,
)
obj_retriever = obj_index.as_retriever(similarity_top_k=3)

# Use the object retriever to dynamically find the
# right tables to use based on the user's query
query_engine_table_retrieval = SQLTableRetrieverQueryEngine(
    sql_database, obj_retriever
)

# --- Example Usage for Table Retrieval ---
print("\n--- Testing Text-to-SQL with Query-Time Table Retrieval ---")
query_1 = "In 2010, what was the number of male inmates?"
response_1 = query_engine_table_retrieval.query(query_1)
print(f"Query: {query_1}")
print(f"Response: {response_1}\n")
print(f"Generated SQL: {response_1.metadata['sql_query']}\n")


--- Testing Text-to-SQL with Query-Time Table Retrieval ---
Query: In 2010, what was the number of male inmates?
Response: In 2010, there were 10,156 male inmates in the convicted penal population.

Generated SQL: SELECT number_of_population
FROM convicted_penal_population_by_gender
WHERE year = 2010 AND population_by_gender = 'Male'
ORDER BY number_of_population DESC;



## Query Time Row Retrieval

In [6]:
# -Create Vector Indices for Rows in Each Table
table_names = inspector.get_table_names()
table_row_indices: Dict[str, VectorStoreIndex] = {}
table_row_query_engines: Dict[str, RetrieverQueryEngine] = {}

print("\n--- Starting Row-Level Indexing for Each Table ---")
for table_name in table_names:
    print(f"Processing table: {table_name}")
    # Read table into pandas DataFrame
    df = pd.read_sql_table(table_name, engine)

    # Create TextNode objects for each row
    row_nodes: List[TextNode] = []
    for i, row in df.iterrows():
        # Combine all columns of the row into a single text string
        row_text = " | ".join(map(str, row.values))
        node = TextNode(
            text=f"Row for table '{table_name}': {row_text}",
            metadata={"table_name": table_name, "row_index": i},
        )
        row_nodes.append(node)

    # Create a VectorStoreIndex from the row nodes
    row_index = VectorStoreIndex(row_nodes)
    table_row_indices[table_name] = row_index

    # Create a query engine for this table's rows
    table_row_query_engines[table_name] = row_index.as_query_engine(
        similarity_top_k=5
    )
print("--- Row-Level Indexing Complete ---\n")

# Create QueryEngineTools for Each Table's Row Data
query_engine_tools: List[QueryEngineTool] = []
for table_name, query_engine in table_row_query_engines.items():
    tool_metadata = f"This tool provides access to the rows of the '{table_name}' table. Use it to find specific values or examples within the table."
    tool = QueryEngineTool.from_defaults(
        query_engine=query_engine,
        name=f"row_retriever_{table_name}",
        description=tool_metadata,
    )
    query_engine_tools.append(tool)

# This engine combines both table retrieval and row retrieval.
# obj_retriever` finds the right tables
# tools (row retrievers) help find the right values within those tables

final_query_engine = SQLTableRetrieverQueryEngine(
    sql_database,
    obj_retriever,
    tools=query_engine_tools, # Tools for row-level retrieval
)

# --- Example Usage for Row Retrieval ---
print("--- Testing Text-to-SQL with Query-Time Row-Level Retrieval ---")
query_2 = "In the table for education level, what was the population for those with 'No Formal Education / Lower Primary' in 2021?"
response_2 = final_query_engine.query(query_2)
print(f"Query: {query_2}")
print(f"Response: {response_2}\n")
print(f"Generated SQL: {response_2.metadata['sql_query']}")


--- Starting Row-Level Indexing for Each Table ---
Processing table: convicted_penal_population_by_age_group_2006_2020
Processing table: convicted_penal_population_by_age_group_2020_onwards
Processing table: convicted_penal_population_by_age_group_and_offence_group_2006_2020
Processing table: convicted_penal_population_by_age_group_and_offence_group_2020_onwards
Processing table: convicted_penal_population_by_education_level
Processing table: convicted_penal_population_by_gender
Processing table: convicted_penal_population_by_gender_and_offence_group
Processing table: convicted_penal_population_by_offence_group
--- Row-Level Indexing Complete ---

--- Testing Text-to-SQL with Query-Time Row-Level Retrieval ---
Query: In the table for education level, what was the population for those with 'No Formal Education / Lower Primary' in 2021?
Response: The query did not return any results for the population with 'No Formal Education / Lower Primary' in 2021 in the table for education level.



## Benchmarking

In [10]:
# --- Configuration ---
relative_benchmark_path = os.getenv("SQL_BENCHMARK_DATASET_DIR")
if not relative_benchmark_path:
    raise ValueError("SQL_BENCHMARK_DATASET_DIR not set in .env")

BENCHMARK_FILE_PATH = os.path.join(project_root, relative_benchmark_path)
OUTPUT_FILENAME = "(llama3.1)adv_sql_benchmark_results.csv"
OUTPUT_FILE_PATH = os.path.join(os.getcwd(), OUTPUT_FILENAME)

# --- Load Data ---
print(f"Loading benchmark data from {BENCHMARK_FILE_PATH}...")
try:
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH)
except UnicodeDecodeError:
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH, encoding="latin1")

# Uncomment to run a smaller test
# benchmark_df = benchmark_df.head(3)

# --- Prepare DataFrame for Ragas ---
# Rename gt_answer to ground_truth for Ragas answer metrics
if 'gt_answer' in benchmark_df.columns:
    benchmark_df['gt_answer'] = benchmark_df['gt_answer'].fillna('')
    benchmark_df = benchmark_df.rename(columns={"gt_answer": "ground_truth"})

# Prepare ground_truths for RAG metrics (though not used by DataCompy)
# and gt_query for DataCompy reference execution
if 'gt_query' in benchmark_df.columns:
    benchmark_df['gt_query'] = benchmark_df['gt_query'].fillna('')
    benchmark_df = benchmark_df.rename(columns={"gt_query": "ground_truths"})
    benchmark_df['ground_truths_list'] = benchmark_df['ground_truths'].apply(lambda x: [x] if isinstance(x, str) else [])
else:
    raise ValueError("'gt_query' column not found in the benchmark file.")

print(f"Loaded {len(benchmark_df)} question-answer pairs for evaluation.")

# Get the database schema using the inspector object from your setup (For LLMSQLEquivalence)
inspector = inspect(engine)
schema_str_list = []
for table_name in inspector.get_table_names():
    schema_str_list.append(f"Table {table_name}:")
    for column in inspector.get_columns(table_name):
        schema_str_list.append(f"  - {column['name']}: {column['type']}")
schema_context = "\n".join(schema_str_list)

# --- Generate Predictions and Collect Data for All Metrics ---
print("--- Running pipeline and collecting data for evaluation ---")
results_data = []
for index, row in tqdm(benchmark_df.iterrows(), total=benchmark_df.shape[0]):
    question = row['question']
    ground_truth_sql = row['ground_truths'] # This is the raw SQL string

    final_response = final_query_engine.query(question)
    generated_sql = final_response.metadata.get('sql_query', 'No SQL Query Generated')
    generated_answer = str(final_response)
    contexts = [node.get_content() for node in final_response.source_nodes]

    # --- Data for DataCompyScore: Execute both SQL queries ---
    predicted_csv = ""
    reference_csv = ""
    sql_error_log = "OK"

    try:
        if generated_sql != 'No SQL Query Generated':
            predicted_df = pd.read_sql_query(generated_sql, engine)
            if predicted_df.empty:
                sql_error_log = "Generated SQL returned an empty result."
            else:
                predicted_csv = predicted_df.to_csv(index=False)
        else:
            sql_error_log = "Pipeline did not generate SQL."

        if ground_truth_sql:
            reference_df = pd.read_sql_query(ground_truth_sql, engine)
            if reference_df.empty:
                sql_error_log = "Ground truth SQL returned an empty result."
            else:
                reference_csv = reference_df.to_csv(index=False)
        else:
            sql_error_log = "Ground truth SQL is missing."
            
    except Exception as e:
        print(f"!!! SQL EXECUTION FAILED for question: '{question}'")
        print(f"!!! DATABASE ERROR: {e}")
        print("-" * 20)
        sql_error_log = str(e)

    results_data.append({
        "question": question,
        "answer": generated_answer,
        "contexts": contexts,
        "ground_truth": row.get('ground_truth'),
        "ground_truths": row.get('ground_truths_list'),
        "gt_query_str": ground_truth_sql,
        "generated_sql": generated_sql,
        "predicted_csv_output": predicted_csv,
        "reference_csv_output": reference_csv,
        "reference_contexts": [schema_context],
        "sql_execution_error": sql_error_log  # Add the error to the results
    })

results_df = pd.DataFrame(results_data)
ragas_dataset = Dataset.from_pandas(results_df)


# --- Instantiate Metrics ---
rag_metrics = [answer_relevancy, faithfulness, context_precision, context_recall]
datacompy_metric = DataCompyScore()
llm_sql_metric = LLMSQLEquivalence()

# --- Run All Evaluations ---
print("Evaluating RAG metrics...")
rag_result = evaluate(dataset=ragas_dataset, metrics=rag_metrics)
print("Evaluating DataCompyScore...")
datacompy_result = evaluate(dataset=ragas_dataset, metrics=[datacompy_metric], column_map={"response": "predicted_csv_output", "reference": "reference_csv_output"})
print("Evaluating LLMSQLEquivalence...")
llm_sql_result = evaluate(dataset=ragas_dataset, metrics=[llm_sql_metric], column_map={"response": "generated_sql", "reference": "gt_query_str", "reference_contexts": "reference_contexts"})

# --- Format and Save Final Results ---
final_df = results_df.copy()

all_score_dfs = {
    'rag': rag_result,
    'datacompy': datacompy_result,
    'llm_sql': llm_sql_result
}
all_score_cols = []

for name, result in all_score_dfs.items():
    scores_df = result.to_pandas()
    new_cols = [col for col in scores_df.columns if col not in final_df.columns]
    if new_cols:
        final_df = final_df.join(scores_df[new_cols])
        all_score_cols.extend(new_cols)

# --- Print Overall Performance Metrics ---
print("\n--- Overall Performance Metrics ---")
print(final_df[all_score_cols].mean(numeric_only=True))
print("----------------------------------")

# Replace blanks and NaN scores with -1
metric_columns_to_keep = [
    'answer_relevancy', 
    'faithfulness', 
    'context_precision', 
    'context_recall', 
    'data_compare_score(mode=rows)', 
    'llm_sql_equivalence_with_reference'
]

# Define the base text columns to keep
text_columns_to_keep = [
    "question", "ground_truth", "gt_query_str", "answer", "generated_sql"
]

# Combine the lists and filter for only columns that actually exist in your DataFrame.
# This prevents errors if a metric failed to run and add its column.
columns_to_keep = text_columns_to_keep + [col for col in metric_columns_to_keep if col in final_df.columns]

# Create the clean DataFrame with ONLY the desired columns
final_df_clean = final_df[columns_to_keep].copy()

# Rename columns for better readability in the final CSV file
final_df_clean.rename(columns={"ground_truth": "gt_answer", "gt_query_str": "gt_query", "answer": "generated_answer"}, inplace=True)

# Save the final, clean DataFrame
final_df_clean.to_csv(OUTPUT_FILE_PATH, index=False)
print(f"\n✅ Clean benchmark results saved to {OUTPUT_FILE_PATH}")

Loading benchmark data from /mnt/batch/tasks/shared/LS_root/mounts/clusters/joo-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/Datasets/Benchmark Dataset/sql_benchmark.csv...
Loaded 100 question-answer pairs for evaluation.
--- Running pipeline and collecting data for evaluation ---


  0%|                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          | 0/100 [00:00<?, ?it/s]

 14%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          | 14/100 [00:41<04:16,  2.98s/it]

!!! SQL EXECUTION FAILED for question: 'How many female inmates were convicted for 'Commercial Crimes' in 2020?'
!!! DATABASE ERROR: (sqlite3.OperationalError) no such column: population_by_crime
[SQL: SELECT number_of_population
FROM convicted_penal_population_by_gender
WHERE year = 2020 AND population_by_gender = 'Female' AND population_by_crime = 'Commercial Crimes']
(Background on this error at: https://sqlalche.me/e/20/e3q8)
--------------------


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [04:13<00:00,  2.54s/it]


...

Evaluating DataCompyScore...


Evaluating:   0%|                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              | 0/100 [00:00<?, ?it/s]/

Evaluating LLMSQLEquivalence...


Evaluating: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:26<00:00,  3.72it/s]



--- Overall Performance Metrics ---
answer_relevancy                      0.867378
faithfulness                          0.081667
context_precision                     0.750000
context_recall                        0.760000
data_compare_score(mode=rows)         0.989583
llm_sql_equivalence_with_reference    0.460000
dtype: float64
----------------------------------

✅ Clean benchmark results saved to /mnt/batch/tasks/shared/LS_root/mounts/clusters/joo-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/(llama3.1)adv_sql_benchmark_results.csv


## Benchmarking Table Retrieval

In [8]:
# --- Configuration ---
relative_benchmark_path = os.getenv("SQL_BENCHMARK_DATASET_DIR")
if not relative_benchmark_path:
    raise ValueError("SQL_BENCHMARK_DATASET_DIR not set in .env")

BENCHMARK_FILE_PATH = os.path.join(project_root, relative_benchmark_path)
OUTPUT_FILENAME = "(table)adv_sql_benchmark_results.csv"
OUTPUT_FILE_PATH = os.path.join(os.getcwd(), OUTPUT_FILENAME)

# --- Load Data ---
print(f"Loading benchmark data from {BENCHMARK_FILE_PATH}...")
try:
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH)
except UnicodeDecodeError:
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH, encoding="latin1")

# Uncomment to run a smaller test
# benchmark_df = benchmark_df.head(3)

# --- Prepare DataFrame for Ragas ---
# Rename gt_answer to ground_truth for Ragas answer metrics
if 'gt_answer' in benchmark_df.columns:
    benchmark_df['gt_answer'] = benchmark_df['gt_answer'].fillna('')
    benchmark_df = benchmark_df.rename(columns={"gt_answer": "ground_truth"})

# Prepare ground_truths for RAG metrics (though not used by DataCompy)
# and gt_query for DataCompy reference execution
if 'gt_query' in benchmark_df.columns:
    benchmark_df['gt_query'] = benchmark_df['gt_query'].fillna('')
    benchmark_df = benchmark_df.rename(columns={"gt_query": "ground_truths"})
    benchmark_df['ground_truths_list'] = benchmark_df['ground_truths'].apply(lambda x: [x] if isinstance(x, str) else [])
else:
    raise ValueError("'gt_query' column not found in the benchmark file.")

print(f"Loaded {len(benchmark_df)} question-answer pairs for evaluation.")

# Get the database schema using the inspector object from your setup (For LLMSQLEquivalence)
inspector = inspect(engine)
schema_str_list = []
for table_name in inspector.get_table_names():
    schema_str_list.append(f"Table {table_name}:")
    for column in inspector.get_columns(table_name):
        schema_str_list.append(f"  - {column['name']}: {column['type']}")
schema_context = "\n".join(schema_str_list)

# --- Generate Predictions and Collect Data for All Metrics ---
print("--- Running pipeline and collecting data for evaluation ---")
results_data = []
for index, row in tqdm(benchmark_df.iterrows(), total=benchmark_df.shape[0]):
    question = row['question']
    ground_truth_sql = row['ground_truths'] # This is the raw SQL string

    final_response = query_engine_table_retrieval.query(question)
    generated_sql = final_response.metadata.get('sql_query', 'No SQL Query Generated')
    generated_answer = str(final_response)
    contexts = [node.get_content() for node in final_response.source_nodes]

    # --- Data for DataCompyScore: Execute both SQL queries ---
    predicted_csv = ""
    reference_csv = ""
    sql_error_log = "OK"

    try:
        if generated_sql != 'No SQL Query Generated':
            predicted_df = pd.read_sql_query(generated_sql, engine)
            if predicted_df.empty:
                sql_error_log = "Generated SQL returned an empty result."
            else:
                predicted_csv = predicted_df.to_csv(index=False)
        else:
            sql_error_log = "Pipeline did not generate SQL."

        if ground_truth_sql:
            reference_df = pd.read_sql_query(ground_truth_sql, engine)
            if reference_df.empty:
                sql_error_log = "Ground truth SQL returned an empty result."
            else:
                reference_csv = reference_df.to_csv(index=False)
        else:
            sql_error_log = "Ground truth SQL is missing."
            
    except Exception as e:
        print(f"!!! SQL EXECUTION FAILED for question: '{question}'")
        print(f"!!! DATABASE ERROR: {e}")
        print("-" * 20)
        sql_error_log = str(e)

    results_data.append({
        "question": question,
        "answer": generated_answer,
        "contexts": contexts,
        "ground_truth": row.get('ground_truth'),
        "ground_truths": row.get('ground_truths_list'),
        "gt_query_str": ground_truth_sql,
        "generated_sql": generated_sql,
        "predicted_csv_output": predicted_csv,
        "reference_csv_output": reference_csv,
        "reference_contexts": [schema_context],
        "sql_execution_error": sql_error_log  # Add the error to the results
    })

results_df = pd.DataFrame(results_data)
ragas_dataset = Dataset.from_pandas(results_df)


# --- Instantiate Metrics ---
rag_metrics = [answer_relevancy, faithfulness, context_precision, context_recall]
datacompy_metric = DataCompyScore()
llm_sql_metric = LLMSQLEquivalence()

# --- Run All Evaluations ---
print("Evaluating RAG metrics...")
rag_result = evaluate(dataset=ragas_dataset, metrics=rag_metrics)
print("Evaluating DataCompyScore...")
datacompy_result = evaluate(dataset=ragas_dataset, metrics=[datacompy_metric], column_map={"response": "predicted_csv_output", "reference": "reference_csv_output"})
print("Evaluating LLMSQLEquivalence...")
llm_sql_result = evaluate(dataset=ragas_dataset, metrics=[llm_sql_metric], column_map={"response": "generated_sql", "reference": "gt_query_str", "reference_contexts": "reference_contexts"})

# --- Format and Save Final Results ---
final_df = results_df.copy()

all_score_dfs = {
    'rag': rag_result,
    'datacompy': datacompy_result,
    'llm_sql': llm_sql_result
}
all_score_cols = []

for name, result in all_score_dfs.items():
    scores_df = result.to_pandas()
    new_cols = [col for col in scores_df.columns if col not in final_df.columns]
    if new_cols:
        final_df = final_df.join(scores_df[new_cols])
        all_score_cols.extend(new_cols)

# --- Print Overall Performance Metrics ---
print("\n--- Overall Performance Metrics ---")
print(final_df[all_score_cols].mean(numeric_only=True))
print("----------------------------------")

# Replace blanks and NaN scores with -1
all_score_cols = [col for col in final_df.columns if col not in ['question', 'ground_truth', 'gt_query_str', 'answer', 'generated_sql']]
for col in all_score_cols:
    # Ensure column is numeric, coercing errors to NaN
    final_df[col] = pd.to_numeric(final_df[col], errors='coerce')
final_df[all_score_cols] = final_df[all_score_cols].fillna(-1)
print("✅ Replaced all blank/NaN metric scores with -1.")

# --- Prepare DataFrame for Final CSV Output ---
columns_to_keep = [
    "question", "ground_truth", "gt_query_str", "answer", "generated_sql"
] + all_score_cols
final_df_clean = final_df[columns_to_keep].copy()
final_df_clean.rename(columns={"ground_truth": "gt_answer", "gt_query_str": "gt_query", "answer": "generated_answer"}, inplace=True)

final_df_clean.to_csv(OUTPUT_FILE_PATH, index=False)
print(f"\nClean benchmark results saved to {OUTPUT_FILE_PATH}")

Loading benchmark data from /mnt/batch/tasks/shared/LS_root/mounts/clusters/joo-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/Datasets/Benchmark Dataset/sql_benchmark.csv...
Loaded 100 question-answer pairs for evaluation.
--- Running pipeline and collecting data for evaluation ---


 14%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          | 14/100 [00:33<03:10,  2.21s/it]

!!! SQL EXECUTION FAILED for question: 'How many female inmates were convicted for 'Commercial Crimes' in 2020?'
!!! DATABASE ERROR: (sqlite3.OperationalError) no such column: population_by_age_group
[SQL: SELECT number_of_population
FROM convicted_penal_population_by_gender
WHERE year = 2020 AND population_by_gender = 'Female' AND population_by_age_group = 'Commercial Crimes'
ORDER BY number_of_population DESC;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
--------------------


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [04:05<00:00,  2.46s/it]


Evaluating RAG metrics...


...

Evaluating DataCompyScore...


Evaluating:   0%|                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              | 0/100 [00:00<?, ?it/s]/

Evaluating LLMSQLEquivalence...


Evaluating: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:28<00:00,  3.56it/s]



--- Overall Performance Metrics ---
answer_relevancy                      0.920452
faithfulness                          0.082500
context_precision                     0.780000
context_recall                        0.770000
data_compare_score(mode=rows)         0.977595
llm_sql_equivalence_with_reference    0.450000
dtype: float64
----------------------------------
✅ Replaced all blank/NaN metric scores with -1.

Clean benchmark results saved to /mnt/batch/tasks/shared/LS_root/mounts/clusters/joo-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/(table)adv_sql_benchmark_results.csv


## Benchmarking Row Retrieval

In [9]:
# --- Create Engine for Row-Aware Retrieval ---
print("Creating a Text-to-SQL engine that uses table schemas AND sample rows for context...")
query_engine_row_aware = NLSQLTableQueryEngine(
    sql_database=sql_database,
    include_sample_rows_in_table_info=True,  # Default behavior, but explicit here
    llm=llm, # Assumes 'llm' is defined in your notebook
)
print("✅ Engine created.")

# --- Configuration ---
relative_benchmark_path = os.getenv("SQL_BENCHMARK_DATASET_DIR")
if not relative_benchmark_path:
    raise ValueError("SQL_BENCHMARK_DATASET_DIR not set in .env")

BENCHMARK_FILE_PATH = os.path.join(project_root, relative_benchmark_path)
OUTPUT_FILENAME = "(row)adv_sql_benchmark_results.csv"
OUTPUT_FILE_PATH = os.path.join(os.getcwd(), OUTPUT_FILENAME)

# --- Load Data ---
print(f"Loading benchmark data from {BENCHMARK_FILE_PATH}...")
try:
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH)
except UnicodeDecodeError:
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH, encoding="latin1")

# Uncomment to run a smaller test
# benchmark_df = benchmark_df.head(3)

# --- Prepare DataFrame for Ragas ---
# Rename gt_answer to ground_truth for Ragas answer metrics
if 'gt_answer' in benchmark_df.columns:
    benchmark_df['gt_answer'] = benchmark_df['gt_answer'].fillna('')
    benchmark_df = benchmark_df.rename(columns={"gt_answer": "ground_truth"})

# Prepare ground_truths for RAG metrics (though not used by DataCompy)
# and gt_query for DataCompy reference execution
if 'gt_query' in benchmark_df.columns:
    benchmark_df['gt_query'] = benchmark_df['gt_query'].fillna('')
    benchmark_df = benchmark_df.rename(columns={"gt_query": "ground_truths"})
    benchmark_df['ground_truths_list'] = benchmark_df['ground_truths'].apply(lambda x: [x] if isinstance(x, str) else [])
else:
    raise ValueError("'gt_query' column not found in the benchmark file.")

print(f"Loaded {len(benchmark_df)} question-answer pairs for evaluation.")

# Get the database schema using the inspector object from your setup (For LLMSQLEquivalence)
inspector = inspect(engine)
schema_str_list = []
for table_name in inspector.get_table_names():
    schema_str_list.append(f"Table {table_name}:")
    for column in inspector.get_columns(table_name):
        schema_str_list.append(f"  - {column['name']}: {column['type']}")
schema_context = "\n".join(schema_str_list)

# --- Generate Predictions and Collect Data for All Metrics ---
print("--- Running pipeline and collecting data for evaluation ---")
results_data = []
for index, row in tqdm(benchmark_df.iterrows(), total=benchmark_df.shape[0]):
    question = row['question']
    ground_truth_sql = row['ground_truths'] # This is the raw SQL string

    final_response = query_engine_row_aware.query(question)
    generated_sql = final_response.metadata.get('sql_query', 'No SQL Query Generated')
    generated_answer = str(final_response)
    contexts = [node.get_content() for node in final_response.source_nodes]

    # --- Data for DataCompyScore: Execute both SQL queries ---
    predicted_csv = ""
    reference_csv = ""
    sql_error_log = "OK"

    try:
        if generated_sql != 'No SQL Query Generated':
            predicted_df = pd.read_sql_query(generated_sql, engine)
            if predicted_df.empty:
                sql_error_log = "Generated SQL returned an empty result."
            else:
                predicted_csv = predicted_df.to_csv(index=False)
        else:
            sql_error_log = "Pipeline did not generate SQL."

        if ground_truth_sql:
            reference_df = pd.read_sql_query(ground_truth_sql, engine)
            if reference_df.empty:
                sql_error_log = "Ground truth SQL returned an empty result."
            else:
                reference_csv = reference_df.to_csv(index=False)
        else:
            sql_error_log = "Ground truth SQL is missing."
            
    except Exception as e:
        print(f"!!! SQL EXECUTION FAILED for question: '{question}'")
        print(f"!!! DATABASE ERROR: {e}")
        print("-" * 20)
        sql_error_log = str(e)

    results_data.append({
        "question": question,
        "answer": generated_answer,
        "contexts": contexts,
        "ground_truth": row.get('ground_truth'),
        "ground_truths": row.get('ground_truths_list'),
        "gt_query_str": ground_truth_sql,
        "generated_sql": generated_sql,
        "predicted_csv_output": predicted_csv,
        "reference_csv_output": reference_csv,
        "reference_contexts": [schema_context],
        "sql_execution_error": sql_error_log  # Add the error to the results
    })

results_df = pd.DataFrame(results_data)
ragas_dataset = Dataset.from_pandas(results_df)


# --- Instantiate Metrics ---
rag_metrics = [answer_relevancy, faithfulness, context_precision, context_recall]
datacompy_metric = DataCompyScore()
llm_sql_metric = LLMSQLEquivalence()

# --- Run All Evaluations ---
print("Evaluating RAG metrics...")
rag_result = evaluate(dataset=ragas_dataset, metrics=rag_metrics)
print("Evaluating DataCompyScore...")
datacompy_result = evaluate(dataset=ragas_dataset, metrics=[datacompy_metric], column_map={"response": "predicted_csv_output", "reference": "reference_csv_output"})
print("Evaluating LLMSQLEquivalence...")
llm_sql_result = evaluate(dataset=ragas_dataset, metrics=[llm_sql_metric], column_map={"response": "generated_sql", "reference": "gt_query_str", "reference_contexts": "reference_contexts"})

# --- Format and Save Final Results ---
final_df = results_df.copy()

all_score_dfs = {
    'rag': rag_result,
    'datacompy': datacompy_result,
    'llm_sql': llm_sql_result
}
all_score_cols = []

for name, result in all_score_dfs.items():
    scores_df = result.to_pandas()
    new_cols = [col for col in scores_df.columns if col not in final_df.columns]
    if new_cols:
        final_df = final_df.join(scores_df[new_cols])
        all_score_cols.extend(new_cols)

# --- Print Overall Performance Metrics ---
print("\n--- Overall Performance Metrics ---")
print(final_df[all_score_cols].mean(numeric_only=True))
print("----------------------------------")

# Replace blanks and NaN scores with -1
all_score_cols = [col for col in final_df.columns if col not in ['question', 'ground_truth', 'gt_query_str', 'answer', 'generated_sql']]
for col in all_score_cols:
    # Ensure column is numeric, coercing errors to NaN
    final_df[col] = pd.to_numeric(final_df[col], errors='coerce')
final_df[all_score_cols] = final_df[all_score_cols].fillna(-1)
print("✅ Replaced all blank/NaN metric scores with -1.")

# --- Prepare DataFrame for Final CSV Output ---
columns_to_keep = [
    "question", "ground_truth", "gt_query_str", "answer", "generated_sql"
] + all_score_cols
final_df_clean = final_df[columns_to_keep].copy()
final_df_clean.rename(columns={"ground_truth": "gt_answer", "gt_query_str": "gt_query", "answer": "generated_answer"}, inplace=True)

final_df_clean.to_csv(OUTPUT_FILE_PATH, index=False)
print(f"\nClean benchmark results saved to {OUTPUT_FILE_PATH}")

Creating a Text-to-SQL engine that uses table schemas AND sample rows for context...
✅ Engine created.
Loading benchmark data from /mnt/batch/tasks/shared/LS_root/mounts/clusters/joo-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/Datasets/Benchmark Dataset/sql_benchmark.csv...
Loaded 100 question-answer pairs for evaluation.
--- Running pipeline and collecting data for evaluation ---


  0%|                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          | 0/100 [00:00<?, ?it/s]S

!!! SQL EXECUTION FAILED for question: 'In 2014, how many inmates aged 'Below 21' were convicted for 'Property Crimes'?'
!!! DATABASE ERROR: (sqlite3.OperationalError) no such table: convicted_penal_population_by_main_offence_group
[SQL: SELECT T1.number_of_population FROM convicted_penal_population_by_age_group_and_offence_group_2006_2020 AS T1 INNER JOIN convicted_penal_population_by_main_offence_group AS T2 ON T1.population_by_main_offence_group = T2.population_by_main_offence_group WHERE T1.year = 2014 AND T1.population_by_age_group = 'Below 21']
(Background on this error at: https://sqlalche.me/e/20/e3q8)
--------------------


Setting `pad_token_id` to `eos_token_id`:26 for open-end generation.
 17%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

!!! SQL EXECUTION FAILED for question: 'In 2017, how many inmates aged '21-30 yrs' were convicted for 'Property Crimes'?'
!!! DATABASE ERROR: (sqlite3.OperationalError) no such column: T2.population_by_main_offence_group
[SQL: SELECT T1.number_of_population FROM convicted_penal_population_by_age_group_2006_2020 AS T1 INNER JOIN convicted_penal_population_by_age_group_2020_onwards AS T2 ON T1.year = T2.year WHERE T1.year = 2019 AND T2.population_by_main_offence_group = 'Violent Crimes']
(Background on this error at: https://sqlalche.me/e/20/e3q8)
--------------------


...

!!! SQL EXECUTION FAILED for question: 'What was the number of '19 & below' inmates for 'Crimes Against Public Order' in 2020?'
!!! DATABASE ERROR: (sqlite3.OperationalError) no such table: convicted_penal_population_by_main_offence_group_2020_onwards
[SQL: SELECT T1.number_of_population FROM convicted_penal_population_by_age_group_and_offence_group_2020_onwards AS T1 INNER JOIN convicted_penal_population_by_main_offence_group_2020_onwards AS T2 ON T1.population_by_main_offence_group = T2.population_by_main_offence_group WHERE T1.year = 2020 AND T1.population_by_age_group = '19 & below' AND T2.population_by_main_offence_group = 'Crimes Against Public Order']
(Background on this error at: https://sqlalche.me/e/20/e3q8)
--------------------


Setting `pad_token_id` to `eos_token_id`:26 for open-end generation.
 31%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

!!! SQL EXECUTION FAILED for question: 'In 2016, how many male inmates were convicted for 'Crimes Against Public Order'?'
!!! DATABASE ERROR: (sqlite3.OperationalError) near "SQL": syntax error
[SQL: SELECT T1.number_of_population FROM convicted_penal_population_by_gender AS T1 INNER JOIN convicted_penal_population_by_gender_and_offence_group AS T2 ON T1.year = T2.year AND T1.population_by_gender = T2.population_by_gender WHERE T1.year = 2016 AND T2.population_by_gender = 'Male' AND T2.population_by_main_offence_group = 'Crimes Against Public Order'
SQL]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
--------------------


Setting `pad_token_id` to `eos_token_id`:26 for open-end generation.
 50%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌                                                                                                                                                                                                                                                                                                                                                                                                                                                             

KeyboardInterrupt: 